In [2]:
# ============================================================
# ToolPathGen — Cell 1
# Imports, reproducibility, paths and visualisation backend
# ============================================================

import sys
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pyvista as pv

from scipy.spatial import cKDTree


# ------------------------------------------------------------
# Project modules
# ------------------------------------------------------------
import numpy as np
from collections import Counter

from Utils.EulerGraph import (
    EulerCCFToolpath,
    calculate_uniform_even_fibre_count,
    compute_edge_normals_from_cad,
    generate_surface_offset_passes,
    coerce_surface_mesh,
)
import importlib
import Utils.EulerGraph as EulerGraph

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer import CADVisualizer
from Utils.CADDomainVisualizer import CADDomainVisualizer

from Decoder_CLasses.ContinuousVoronoiDecoder import (
    ContinuousVoronoiDecoder,
)
from Decoder_CLasses.NearUniformHoneycombBaseline import (
    NearUniformHoneycombBaseline,
)
from Decoder_CLasses.IntrinsicSurfaceHoneycombBaseline import (
    IntrinsicSurfaceHoneycombBaseline,
)

from Utils.EulerGraph import (
    EulerCCFToolpath,
    ManufacturingGeometryError,
    ManufacturingInfeasibilityError,
    calculate_uniform_even_fibre_count,
    coerce_surface_mesh,
    compute_edge_normals_from_cad,
    generate_surface_offset_passes,
    prepare_manufacturing_physical_graph,
)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

BASE = Path.cwd()
TEST_PARTS_DIR = BASE / "Testparts"
OUTPUT_DIR = BASE / "ToolPath_Output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Code directory      :", BASE)
print("STEP-parts directory:", TEST_PARTS_DIR)
print("Output directory    :", OUTPUT_DIR)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Torch device        :", device)


# ------------------------------------------------------------
# PyVista notebook backend
# ------------------------------------------------------------

def setup_pyvista():
    is_mac = sys.platform == "darwin"

    if is_mac:
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.OFF_SCREEN = False
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception as exc:
            warnings.warn(
                f"Interactive PyVista unavailable: {exc}. "
                "Using the static backend."
            )
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print("PyVista backend     :", backend)


setup_pyvista()

%matplotlib inline


# ------------------------------------------------------------
# Optional seed generator for a new Voronoi example
# ------------------------------------------------------------

def random_seeds_min_dist(
    number_of_seeds,
    min_distance=0.08,
    seed=1,
    max_attempts=10000,
    device="cpu",
):
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)

    seeds = []

    for _ in range(max_attempts):
        if len(seeds) >= number_of_seeds:
            break

        candidate = torch.rand(
            2,
            generator=generator,
            device=device,
        )

        if not seeds:
            seeds.append(candidate)
            continue

        existing = torch.stack(seeds)
        distances = torch.linalg.norm(
            existing - candidate[None, :],
            dim=1,
        )

        if torch.min(distances) >= min_distance:
            seeds.append(candidate)

    if len(seeds) != number_of_seeds:
        raise RuntimeError(
            f"Generated only {len(seeds)}/{number_of_seeds} seeds. "
            f"Reduce min_distance={min_distance} or increase "
            f"max_attempts={max_attempts}."
        )

    return torch.stack(seeds)


print("\nToolPathGen imports completed successfully.")

Code directory      : /home/arash/HVD_SeedsBase
STEP-parts directory: /home/arash/HVD_SeedsBase/Testparts
Output directory    : /home/arash/HVD_SeedsBase/ToolPath_Output
Torch device        : cuda
PyVista backend     : trame

ToolPathGen imports completed successfully.


In [3]:

def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_surface_density(fields, density_key="rho"):
    face_tensor = fields["face_tensor"]

    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary.
    if triangles.min() == 1:
        triangles = triangles - 1

    # PyVista face format:
    # [3, i, j, k, 3, i, j, k, ...]
    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    mesh = pv.PolyData(points, pv_faces)

    density = to_numpy(
        fields[density_key]
    ).reshape(-1)

    mesh.point_data["density"] = density

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": density_key,
        },
    )

    plotter.add_text(
        f"Surface density: {density_key}",
        font_size=12,
    )

    plotter.show_axes()
    plotter.show()

In [4]:
# ============================================================
# ToolPathGen — Cell 2
# Load CAD surface and generate its triangle mesh
# ============================================================

viz = CADVisualizer()


# ------------------------------------------------------------
# Available STEP models
# ------------------------------------------------------------

SHAPE_PATHS = {
    "CircularSurf1": TEST_PARTS_DIR / "CircularSurf1.stp",
    "Planar": TEST_PARTS_DIR / "Planar.stp",
    "FreeFormSurf3": TEST_PARTS_DIR / "FreeForm3.stp",
    "FreeSharp2": TEST_PARTS_DIR / "FreeSharp2.stp",
    "CircularHoles": TEST_PARTS_DIR / "CircularHoles.stp",
    "SphereTap": TEST_PARTS_DIR / "SphereTap.stp",
    "Helmet": TEST_PARTS_DIR / "Helmet_New.stp",
    "FreeForm_Bend": TEST_PARTS_DIR / "FreeForm_bend.stp",
    "CircBracket_Half": TEST_PARTS_DIR / "CircBracket_Half.stp",
    "HelmetNew_Half": TEST_PARTS_DIR / "HelmetNew_Half.stp",
    "Wing_NACA2412": TEST_PARTS_DIR / "Wing_NACA2412.stp",
}


# ------------------------------------------------------------
# Select the surface
# ------------------------------------------------------------

SHAPE_NAME = "CircularSurf1"
shape_path = SHAPE_PATHS[SHAPE_NAME]

if not shape_path.exists():
    raise FileNotFoundError(
        f"STEP file does not exist:\n{shape_path}"
    )

print("Selected surface:", SHAPE_NAME)
print("STEP file       :", shape_path)


# ------------------------------------------------------------
# Create CAD evaluator and surface mesh
# ------------------------------------------------------------

Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=512,
    mesh_size_scale=0.3,
)

cad_domain, face_mesh = Face_Cad.generate_from_file(
    shape_path
)

face_dimensions = Face_Cad.print_face_info()

if face_dimensions is not None:
    try:
        dx, dy, dz = face_dimensions
        print(
            f"Face dimensions : "
            f"dx={dx:.6g}, dy={dy:.6g}, dz={dz:.6g}"
        )
    except (TypeError, ValueError):
        print("Face information:", face_dimensions)


# ------------------------------------------------------------
# CAD evaluator required by offset and transition generation
# ------------------------------------------------------------

def _as_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def surface_eval_np(uv):
    """
    Evaluate the CAD surface using normalized UV coordinates.

    Xu and Xv are recomputed numerically with respect to the same
    normalized UV coordinates supplied to this function.
    """
    uv = np.asarray(uv, dtype=np.float64).reshape(-1, 2)

    result = dict(
        Face_Cad.eval_uv_norm_batch(
            uv,
            return_inside_mask=True,
        )
    )

    derivative_step = 1.0e-5

    uv_u0 = uv.copy()
    uv_u1 = uv.copy()
    uv_v0 = uv.copy()
    uv_v1 = uv.copy()

    uv_u0[:, 0] = uv[:, 0] - derivative_step
    uv_u1[:, 0] = uv[:, 0] + derivative_step
    uv_v0[:, 1] = uv[:, 1] - derivative_step
    uv_v1[:, 1] = uv[:, 1] + derivative_step

    xyz_u0 = _as_numpy(
        Face_Cad.eval_uv_norm_batch(
            uv_u0,
            return_inside_mask=True,
        )["xyz"]
    )
    xyz_u1 = _as_numpy(
        Face_Cad.eval_uv_norm_batch(
            uv_u1,
            return_inside_mask=True,
        )["xyz"]
    )
    xyz_v0 = _as_numpy(
        Face_Cad.eval_uv_norm_batch(
            uv_v0,
            return_inside_mask=True,
        )["xyz"]
    )
    xyz_v1 = _as_numpy(
        Face_Cad.eval_uv_norm_batch(
            uv_v1,
            return_inside_mask=True,
        )["xyz"]
    )

    du = (uv_u1[:, 0] - uv_u0[:, 0])[:, None]
    dv = (uv_v1[:, 1] - uv_v0[:, 1])[:, None]

    if np.any(du <= 0.0) or np.any(dv <= 0.0):
        raise RuntimeError(
            "Unable to calculate normalized CAD derivatives."
        )

    result["Xu"] = (xyz_u1 - xyz_u0) / du
    result["Xv"] = (xyz_v1 - xyz_v0) / dv

    normal = np.cross(result["Xu"], result["Xv"])
    normal_length = np.linalg.norm(
        normal,
        axis=1,
        keepdims=True,
    )

    if np.any(normal_length <= 1.0e-12):
        raise RuntimeError(
            "Undefined CAD normal found during evaluation."
        )

    result["normal"] = normal / normal_length

    return result

# ------------------------------------------------------------
# Extract mesh for PyVista visualisation
# ------------------------------------------------------------

points = (
    face_mesh["points_xyz"]
    .detach()
    .cpu()
    .numpy()
)

pv_faces = (
    face_mesh["pv_faces"]
    .detach()
    .cpu()
    .numpy()
)

surface_mesh = pv.PolyData(
    points,
    pv_faces,
)


# ------------------------------------------------------------
# Extract clean [V,3] vertices and [F,3] triangular faces
# for toolpath calculations
# ------------------------------------------------------------

surface_vertices, surface_faces = coerce_surface_mesh(
    None,
    None,
    face_mesh=face_mesh,
)


# ------------------------------------------------------------
# Basic mesh verification
# ------------------------------------------------------------

if surface_vertices.ndim != 2 or surface_vertices.shape[1] != 3:
    raise ValueError(
        "surface_vertices must have shape [number_of_vertices, 3]."
    )

if surface_faces.ndim != 2 or surface_faces.shape[1] != 3:
    raise ValueError(
        "surface_faces must contain triangular indices with shape "
        "[number_of_faces, 3]."
    )

if not np.isfinite(surface_vertices).all():
    raise ValueError(
        "Surface mesh contains non-finite coordinates."
    )

print("\nSurface mesh")
print("------------")
print("Vertices       :", len(surface_vertices))
print("Triangle faces :", len(surface_faces))
print("Bounds         :", surface_mesh.bounds)
print("All triangles  :", surface_mesh.is_all_triangles)


# ------------------------------------------------------------
# Visualise
# ------------------------------------------------------------

plotter = pv.Plotter()

plotter.add_mesh(
    surface_mesh,
    color="lightgray",
    show_edges=True,
    edge_color="gray",
    opacity=1.0,
)

plotter.add_axes()
plotter.show()

Selected surface: CircularSurf1
STEP file       : /home/arash/HVD_SeedsBase/Testparts/CircularSurf1.stp
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    :  - Label 'Shapes/Document' (2D)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Circle)
Info    : [ 30%] Meshing curve 2 (BSpline)
Info    : [ 60%] Meshing curve 3 (Circle)
Info    : [ 80%] Meshing curve 4 (BSpline)
Info    : Done meshing 1D (Wall 0.000352777s, CPU 2.4e-05s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Cylinder, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.631309s, CPU 0.621277s)
Info    : 29858 nodes 59718 elements
Info    : Clearing all models and views...
Info    : Done clearing all models and views

=== Active Face Info ===
XYZ bounds:
  X: [-0.000000, 10.000000]  span=10.000000
  Y: [-0.127468, 2.100919]  span=2.228387
  Z: [-2.032622, 1.938721]  span=3.971343

UV bounds:
  U: [-0.039832, 3.264132]
  V: [-5.000000, 5.000000]

Periodic:
  U p

Widget(value='<iframe src="http://localhost:39279/index.html?ui=P_0x72cac8812f80_0&reconnect=auto" class="pyvi…

In [5]:
import importlib
import Utils.EulerGraph as EulerGraphModule

importlib.reload(EulerGraphModule)

generate_surface_offset_passes = (
    EulerGraphModule.generate_surface_offset_passes
)

In [6]:
# ============================================================
# ToolPathGen — Cell 3
# Generate the near-uniform honeycomb reference geometry
# ============================================================

STRUT_THICKNESS = 0.4


# ------------------------------------------------------------
# Create the Voronoi decoder
# ------------------------------------------------------------

decoder = ContinuousVoronoiDecoder(
    Cad_domain=Face_Cad,
    face_mesh=face_mesh,
    n_seeds=100,
    strut_thickness=STRUT_THICKNESS,
    seed_domain_margin=0.8,
)


# ------------------------------------------------------------
# Configure the honeycomb generator
# ------------------------------------------------------------

baseline = NearUniformHoneycombBaseline(
    decoder=decoder,
    cad_domain=Face_Cad,
    face_mesh=face_mesh,

    target_seed_count=None,
    spacing_uv=0.3,

    mode="uv_hex",
    relaxation_steps=0,
    relaxation_factor=0.5,

    include_exterior_support_sites=True,
    support_band_cells=2.5,

    relax_support_sites=True,
    support_relaxation_neighbors=6,
    max_relaxation_displacement_cells=0.75,
    preserve_lattice_topology=True,
)


# ------------------------------------------------------------
# Generate structural fields
# ------------------------------------------------------------

honeycomb_fields = baseline.generate(
    device=device,
    dtype=torch.float64,
    generate_density_fiber=True,
)


# ------------------------------------------------------------
# Verify required toolpath data
# ------------------------------------------------------------

required_keys = [
    "edge_curves_uv",
    "edge_curves_xyz",
    "graph",
    "total_curve_length",
    "total_voronoi_curve_length",
]

missing_keys = [
    key for key in required_keys
    if key not in honeycomb_fields
]

if missing_keys:
    raise KeyError(
        f"Honeycomb generation is missing required fields: "
        f"{missing_keys}"
    )


curves_uv = honeycomb_fields["edge_curves_uv"]
curves_xyz = honeycomb_fields["edge_curves_xyz"]
graph = honeycomb_fields["graph"]

edge_index_tensor = graph["edge_index"]

if edge_index_tensor.ndim != 2:
    raise ValueError(
        "graph['edge_index'] must be a two-dimensional tensor."
    )

if edge_index_tensor.shape[0] == 2:
    number_of_edges = int(edge_index_tensor.shape[1])
elif edge_index_tensor.shape[1] == 2:
    number_of_edges = int(edge_index_tensor.shape[0])
else:
    raise ValueError(
        "graph['edge_index'] must have shape [2,E] or [E,2]."
    )

if curves_uv.shape[0] != number_of_edges:
    raise ValueError(
        "edge_curves_uv and graph edge count do not agree."
    )

if curves_xyz.shape[0] != number_of_edges:
    raise ValueError(
        "edge_curves_xyz and graph edge count do not agree."
    )

if curves_uv.shape[-1] != 2:
    raise ValueError(
        "edge_curves_uv must have final dimension 2."
    )

if curves_xyz.shape[-1] != 3:
    raise ValueError(
        "edge_curves_xyz must have final dimension 3."
    )


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

total_all = (
    honeycomb_fields["total_curve_length"]
    .detach()
    .cpu()
    .item()
)

total_voronoi = (
    honeycomb_fields["total_voronoi_curve_length"]
    .detach()
    .cpu()
    .item()
)

print("Honeycomb generation")
print("--------------------")
print("Physical edges             :", number_of_edges)
print("UV curve shape             :", tuple(curves_uv.shape))
print("XYZ curve shape            :", tuple(curves_xyz.shape))
print("Total curve length         :", total_all)
print("Total Voronoi curve length :", total_voronoi)


# Edge-type diagnostics
if "edge_type" in graph:
    edge_types = graph["edge_type"]

    if hasattr(edge_types, "detach"):
        edge_types = (
            edge_types
            .detach()
            .cpu()
            .numpy()
        )

    edge_type_counts = Counter(
        str(value) for value in np.asarray(edge_types).reshape(-1)
    )

    print("Edge-type counts           :", dict(edge_type_counts))


# Endpoint-stitching diagnostics
graph_diagnostics = graph.get("diagnostics", {})

print(
    "Endpoint gap before stitch:",
    graph_diagnostics.get(
        "max_edge_endpoint_gap_before_stitch",
        "not reported",
    ),
)

print(
    "Endpoint gap after stitch :",
    graph_diagnostics.get(
        "max_edge_endpoint_gap_after_stitch",
        "not reported",
    ),
)


# ------------------------------------------------------------
# Optional binary density only for visualisation
# Do not overwrite the original continuous rho field.
# ------------------------------------------------------------

if "rho" in honeycomb_fields:
    honeycomb_plot_fields = dict(honeycomb_fields)

    honeycomb_plot_fields["rho"] = (
        honeycomb_fields["rho"] > 0.5
    ).to(
        dtype=honeycomb_fields["rho"].dtype,
        device=honeycomb_fields["rho"].device,
    )

    if "show_surface_density" in globals():
        show_surface_density(
            honeycomb_plot_fields,
            density_key="rho",
        )
    else:
        print(
            "show_surface_density() is not currently loaded; "
            "honeycomb generation itself completed successfully."
        )


# Fields selected for toolpath generation
TOOLPATH_FIELDS = honeycomb_fields

print("\nHoneycomb fields are ready for toolpath preparation.")

Honeycomb generation
--------------------
Physical edges             : 63
UV curve shape             : (63, 64, 2)
XYZ curve shape            : (63, 64, 3)
Total curve length         : 91.14985667731314
Total Voronoi curve length : 58.02879207361254
Edge-type counts           : {'0': 29, '1': 14, '3': 2, '4': 18}
Endpoint gap before stitch: 2.220446049250313e-16
Endpoint gap after stitch : 0.0


Widget(value='<iframe src="http://localhost:39279/index.html?ui=P_0x72c7c48aff40_1&reconnect=auto" class="pyvi…


Honeycomb fields are ready for toolpath preparation.


In [7]:
# ============================================================
# ToolPathGen — Cell 3B
# Load an optimized structural model
#
# Run either Cell 3A (honeycomb) or Cell 3B (optimized model),
# not both.
# ============================================================

from Training.MainTrain import (
    load_optimized_shell_function,
    evaluate_optimized_shell_function,
)


# ------------------------------------------------------------
# Available optimized cases
# Keep the matching physical strut thickness with each case.
# ------------------------------------------------------------

OPTIMIZED_CASES = {
    "Planar_40_Sth4": {
        "path": Path(
            "/home/arash/HVD_SeedsBase/Case_Studies/"
            "Planar_40_Sth4_tensile_compression4/"
            "optimized_shell_function.pt"
        ),
        "strut_thickness": 0.4,
        "expected_surface": "Planar",
    },

    "Planar_70_Sth033_BestFeasible": {
        "path": Path(
            "/home/arash/HVD_SeedsBase/Case_Studies/"
            "PlanarPlannar_Seeds-70_Sth033/"
            "BestFeasibleCheckpoint/"
            "optimized_shell_function.pt"
        ),
        "strut_thickness": 0.33,
        "expected_surface": "Planar",
    },
}


# ------------------------------------------------------------
# Select optimized case
# ------------------------------------------------------------

OPTIMIZED_CASE_NAME = "Planar_40_Sth4"

selected_case = OPTIMIZED_CASES[OPTIMIZED_CASE_NAME]

model_path = selected_case["path"]
STRUT_THICKNESS = float(
    selected_case["strut_thickness"]
)
expected_surface = selected_case["expected_surface"]

if not model_path.exists():
    raise FileNotFoundError(
        f"Optimized shell-function file does not exist:\n"
        f"{model_path}"
    )

if SHAPE_NAME != expected_surface:
    raise ValueError(
        f"The optimized case expects surface "
        f"{expected_surface!r}, but Cell 2 loaded "
        f"{SHAPE_NAME!r}."
    )

print("Optimized case :", OPTIMIZED_CASE_NAME)
print("Model path     :", model_path)
print("Surface        :", SHAPE_NAME)
print("Strut thickness:", STRUT_THICKNESS)


# ------------------------------------------------------------
# Load and evaluate the optimized representation
# ------------------------------------------------------------

optimized_function = load_optimized_shell_function(
    str(model_path),
    device=device,
)

optimized_fields = evaluate_optimized_shell_function(
    optimized_function
)


# ------------------------------------------------------------
# Verify required toolpath fields
# ------------------------------------------------------------

required_keys = [
    "edge_curves_uv",
    "edge_curves_xyz",
    "graph",
    "total_curve_length",
    "total_voronoi_curve_length",
]

missing_keys = [
    key for key in required_keys
    if key not in optimized_fields
]

if missing_keys:
    raise KeyError(
        f"Optimized fields are missing required data: "
        f"{missing_keys}"
    )


graph = optimized_fields["graph"]
curves_uv = optimized_fields["edge_curves_uv"]
curves_xyz = optimized_fields["edge_curves_xyz"]
edge_index_tensor = graph["edge_index"]

if edge_index_tensor.ndim != 2:
    raise ValueError(
        "graph['edge_index'] must be two-dimensional."
    )

if edge_index_tensor.shape[0] == 2:
    number_of_edges = int(edge_index_tensor.shape[1])
elif edge_index_tensor.shape[1] == 2:
    number_of_edges = int(edge_index_tensor.shape[0])
else:
    raise ValueError(
        "graph['edge_index'] must have shape [2,E] or [E,2]."
    )

if curves_uv.shape[0] != number_of_edges:
    raise ValueError(
        "Optimized UV-curve count does not match the graph."
    )

if curves_xyz.shape[0] != number_of_edges:
    raise ValueError(
        "Optimized XYZ-curve count does not match the graph."
    )


# ------------------------------------------------------------
# Scalar diagnostics
# ------------------------------------------------------------

def scalar_value(value):
    if hasattr(value, "detach"):
        return float(
            value.detach().cpu().reshape(-1)[0].item()
        )

    return float(
        np.asarray(value).reshape(-1)[0]
    )


total_all = scalar_value(
    optimized_fields["total_curve_length"]
)

total_voronoi = scalar_value(
    optimized_fields["total_voronoi_curve_length"]
)

print("\nOptimized field summary")
print("-----------------------")
print("Physical edges             :", number_of_edges)
print("UV curve shape             :", tuple(curves_uv.shape))
print("XYZ curve shape            :", tuple(curves_xyz.shape))
print("Total curve length         :", total_all)
print("Total Voronoi curve length :", total_voronoi)


# ------------------------------------------------------------
# Optional thresholded density for visualisation only
# ------------------------------------------------------------

if "rho" in optimized_fields:
    optimized_plot_fields = dict(
        optimized_fields
    )

    optimized_plot_fields["rho"] = (
        optimized_fields["rho"] > 0.5
    ).to(
        dtype=optimized_fields["rho"].dtype,
        device=optimized_fields["rho"].device,
    )

    if "show_surface_density" in globals():
        show_surface_density(
            optimized_plot_fields,
            density_key="rho",
        )
    else:
        print(
            "show_surface_density() is not loaded; "
            "the optimized fields were loaded successfully."
        )


# ------------------------------------------------------------
# Select these fields for toolpath generation
# ------------------------------------------------------------

TOOLPATH_FIELDS = honeycomb_fields
TOOLPATH_SOURCE = "optimized"

print("\nOptimized fields are ready for toolpath preparation.")

ValueError: The optimized case expects surface 'Planar', but Cell 2 loaded 'CircularSurf1'.

In [8]:
# ============================================================
# ToolPathGen — Cell 4
# Prepare and deduplicate the physical manufacturing graph
# ============================================================
TOOLPATH_FIELDS = honeycomb_fields
TOOLPATH_SOURCE = "HC"
FIBRE_LINE_WIDTH = 0.05
FIBRE_GAP = 0.025
BOUNDARY_MARGIN = 0.0

selected_fields = TOOLPATH_FIELDS
selected_graph = selected_fields["graph"]


# ------------------------------------------------------------
# Small conversion helper
# ------------------------------------------------------------

def to_numpy(value, dtype=None):
    if value is None:
        return None

    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    else:
        value = np.asarray(value)

    if dtype is not None:
        value = value.astype(dtype, copy=False)

    return value


# ------------------------------------------------------------
# Read the original physical graph
# ------------------------------------------------------------

required_graph_keys = [
    "edge_index",
    "edge_type",
]

missing_graph_keys = [
    key for key in required_graph_keys
    if key not in selected_graph
]

if missing_graph_keys:
    raise KeyError(
        f"Selected graph is missing required metadata: "
        f"{missing_graph_keys}"
    )


original_edge_index = to_numpy(
    selected_graph["edge_index"],
    dtype=np.int64,
)

original_edge_curves_xyz = to_numpy(
    selected_fields["edge_curves_xyz"],
    dtype=np.float64,
)

original_edge_curves_uv = to_numpy(
    selected_fields["edge_curves_uv"],
    dtype=np.float64,
)

original_edge_types = to_numpy(
    selected_graph["edge_type"],
    dtype=np.int64,
).reshape(-1)


# Normalise edge_index to [E,2]
if original_edge_index.ndim != 2:
    raise ValueError(
        "edge_index must have shape [E,2] or [2,E]."
    )

if original_edge_index.shape[1] == 2:
    original_edge_index = original_edge_index.copy()
elif original_edge_index.shape[0] == 2:
    original_edge_index = original_edge_index.T.copy()
else:
    raise ValueError(
        "edge_index must have shape [E,2] or [2,E]."
    )


original_edge_count = len(original_edge_index)

if len(original_edge_curves_xyz) != original_edge_count:
    raise ValueError(
        "XYZ-curve count does not match edge_index."
    )

if len(original_edge_curves_uv) != original_edge_count:
    raise ValueError(
        "UV-curve count does not match edge_index."
    )

if len(original_edge_types) != original_edge_count:
    raise ValueError(
        "edge_type count does not match edge_index."
    )


# ------------------------------------------------------------
# Calculate CAD normals at the exact curve samples
# ------------------------------------------------------------

(
    original_edge_normals_xyz,
    normal_diagnostics,
) = compute_edge_normals_from_cad(
    Face_Cad,
    selected_fields,
    surface_vertices=surface_vertices,
    surface_faces=surface_faces,
    xyz_tolerance=1e-6,
    nozzle_approach="+normal",
)

original_edge_normals_xyz = to_numpy(
    original_edge_normals_xyz,
    dtype=np.float64,
)

if original_edge_normals_xyz.shape != original_edge_curves_xyz.shape:
    raise ValueError(
        "CAD-normal shape does not match XYZ-curve shape: "
        f"{original_edge_normals_xyz.shape} versus "
        f"{original_edge_curves_xyz.shape}."
    )


# ------------------------------------------------------------
# Scale-aware duplicate tolerance
#
# Duplicate candidates must also:
# - share the same unordered graph-node pair;
# - contain one type-3 and one type-4 edge.
# ------------------------------------------------------------

surface_extent = np.ptp(
    surface_vertices,
    axis=0,
)

surface_scale = float(
    np.linalg.norm(surface_extent)
)

COINCIDENT_EDGE_TOLERANCE = max(
    1e-8,
    2e-6 * surface_scale,
)

print(
    "Coincident-edge tolerance:",
    COINCIDENT_EDGE_TOLERANCE,
)


# ------------------------------------------------------------
# Preserve useful per-edge metadata when available
# ------------------------------------------------------------

parallel_edge_metadata = {}

for metadata_name in (
    "edge_seed_pair",
    "edge_seed_pair_original",
    "edge_curve_lengths_xyz",
    "edge_trim_alpha",
):
    if metadata_name in selected_graph:
        parallel_edge_metadata[metadata_name] = to_numpy(
            selected_graph[metadata_name]
        )


# ------------------------------------------------------------
# Remove coincident type-3/type-4 boundary duplicates
# ------------------------------------------------------------

manufacturing_graph = prepare_manufacturing_physical_graph(
    edge_index=original_edge_index,
    edge_curves_xyz=original_edge_curves_xyz,
    edge_types=original_edge_types,
    edge_curves_uv=original_edge_curves_uv,
    edge_normals_xyz=original_edge_normals_xyz,
    strut_thickness=STRUT_THICKNESS,
    coincident_edge_tolerance=COINCIDENT_EDGE_TOLERANCE,
    minimum_manufacturable_edge_length=FIBRE_LINE_WIDTH,
    **parallel_edge_metadata,
)


# ------------------------------------------------------------
# Canonical filtered variables used by all following cells
# ------------------------------------------------------------

edge_index = manufacturing_graph["edge_index"]

edge_curves_xyz = np.asarray(
    manufacturing_graph["edge_curves_xyz"],
    dtype=np.float64,
)

edge_curves_uv = np.asarray(
    manufacturing_graph["edge_curves_uv"],
    dtype=np.float64,
)

edge_normals_xyz = np.asarray(
    manufacturing_graph["edge_normals_xyz"],
    dtype=np.float64,
)

edge_types = np.asarray(
    manufacturing_graph["edge_types"],
    dtype=np.int64,
)

original_physical_edge_ids = np.asarray(
    manufacturing_graph[
        "kept_original_physical_edge_ids"
    ],
    dtype=np.int64,
)

old_to_new_edge_id = manufacturing_graph[
    "old_to_new_edge_id"
]

new_to_old_edge_id = manufacturing_graph[
    "new_to_old_edge_id"
]

manufacturing_graph_diagnostics = manufacturing_graph[
    "diagnostics"
]


# Type 4 represents the retained physical boundary edges
boundary_edge_mask = edge_types == 4


# ------------------------------------------------------------
# Final consistency checks
# ------------------------------------------------------------

filtered_edge_count = len(edge_index)

expected_shape_xyz = (
    filtered_edge_count,
    edge_curves_xyz.shape[1],
    3,
)

expected_shape_uv = (
    filtered_edge_count,
    edge_curves_uv.shape[1],
    2,
)

if edge_curves_xyz.shape != expected_shape_xyz:
    raise ValueError(
        f"Unexpected XYZ-curve shape: "
        f"{edge_curves_xyz.shape}"
    )

if edge_curves_uv.shape != expected_shape_uv:
    raise ValueError(
        f"Unexpected UV-curve shape: "
        f"{edge_curves_uv.shape}"
    )

if edge_normals_xyz.shape != edge_curves_xyz.shape:
    raise ValueError(
        "Filtered normal and XYZ-curve shapes disagree."
    )

if len(original_physical_edge_ids) != filtered_edge_count:
    raise ValueError(
        "Original-edge mapping does not match the "
        "filtered edge count."
    )

if np.any(edge_types == 3):
    raise ManufacturingGeometryError(
        "Type-3 duplicate edges remain after manufacturing "
        "graph preparation."
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

merged_pairs = manufacturing_graph_diagnostics[
    "merged_pairs"
]

print("\nManufacturing graph preparation")
print("-------------------------------")
print("Toolpath source          :", TOOLPATH_SOURCE)
print("Original physical edges  :", original_edge_count)
print("Filtered physical edges  :", filtered_edge_count)
print("Removed duplicate edges  :", len(merged_pairs))
print("Boundary edges, type 4   :", int(boundary_edge_mask.sum()))
print("Filtered edge-type counts:", dict(Counter(edge_types.tolist())))
print("XYZ curve-array shape    :", edge_curves_xyz.shape)
print("UV curve-array shape     :", edge_curves_uv.shape)
print("Normal-array shape       :", edge_normals_xyz.shape)
print("Normal diagnostics       :", normal_diagnostics)


if merged_pairs:
    print("\nMerged coincident pairs")
    print("------------------------")

    for pair in merged_pairs:
        print(
            f"removed original edge "
            f"{pair['removed_original_edge_id']} -> "
            f"retained original edge "
            f"{pair['preserved_original_edge_id']} -> "
            f"filtered edge "
            f"{pair['preserved_filtered_edge_id']} "
            f"(distance="
            f"{pair['geometric_distance']:.8g})"
        )
else:
    print("\nNo coincident type-3/type-4 pairs were found.")


print("\nManufacturing physical graph is ready.")

Coincident-edge tolerance: 2.197610266474671e-05

Manufacturing graph preparation
-------------------------------
Toolpath source          : HC
Original physical edges  : 63
Filtered physical edges  : 61
Removed duplicate edges  : 2
Boundary edges, type 4   : 18
Filtered edge-type counts: {0: 29, 1: 14, 4: 18}
XYZ curve-array shape    : (61, 64, 3)
UV curve-array shape     : (61, 64, 2)
Normal-array shape       : (61, 64, 3)
Normal diagnostics       : {'nozzle_approach': '+normal', 'normal_method': 'cad_derivatives', 'max_reevaluated_xyz_error': 4.74066038746398e-07, 'xyz_tolerance': 1e-06, 'xyz_matches_supplied': True, 'undefined_normal_count': 0, 'undefined_normal_locations': [], 'inconsistent_normal_count': 0, 'inconsistent_normal_locations': []}

Merged coincident pairs
------------------------
removed original edge 8 -> retained original edge 50 -> filtered edge 48 (distance=2.47999e-06)
removed original edge 22 -> retained original edge 52 -> filtered edge 50 (distance=7.1965516e

In [9]:
# ============================================================
# ToolPathGen — Cell 5
# Manufacturing fibre configuration
# ============================================================

FIBRE_LINE_WIDTH = 0.05
FIBRE_GAP = 0.025
BOUNDARY_MARGIN = 0.0

REQUESTED_FIBRES_PER_EDGE = None

fibres_per_edge = calculate_uniform_even_fibre_count(
    strut_thickness=STRUT_THICKNESS,
    fibre_line_width=FIBRE_LINE_WIDTH,
    fibre_gap=FIBRE_GAP,
    boundary_margin=BOUNDARY_MARGIN,
    fibres_per_edge=REQUESTED_FIBRES_PER_EDGE,
    boundary_edge_mask=boundary_edge_mask,
)

centreline_pitch = (
    FIBRE_LINE_WIDTH + FIBRE_GAP
)

required_bundle_width = (
    fibres_per_edge * FIBRE_LINE_WIDTH
    + (fibres_per_edge - 1) * FIBRE_GAP
)

print("Manufacturing fibre configuration")
print("---------------------------------")
print("Strut thickness       :", STRUT_THICKNESS)
print("Fibre line width      :", FIBRE_LINE_WIDTH)
print("Fibre gap             :", FIBRE_GAP)
print("Centreline pitch      :", centreline_pitch)
print("Fibres per edge       :", fibres_per_edge)
print("Required bundle width :", required_bundle_width)

Manufacturing fibre configuration
---------------------------------
Strut thickness       : 0.4
Fibre line width      : 0.05
Fibre gap             : 0.025
Centreline pitch      : 0.07500000000000001
Fibres per edge       : 2
Required bundle width : 0.125


In [10]:
(
    edge_pass_curves_xyz,
    edge_pass_normals_xyz,
    edge_pass_curves_uv,
    edge_pass_offsets,
    offset_diagnostics,
) = generate_surface_offset_passes(
    edge_curves_xyz=edge_curves_xyz,
    fibres_per_edge=fibres_per_edge,
    fibre_line_width=FIBRE_LINE_WIDTH,
    fibre_gap=FIBRE_GAP,
    strut_thickness=STRUT_THICKNESS,
    boundary_margin=BOUNDARY_MARGIN,

    edge_normals_xyz=edge_normals_xyz,
    edge_curves_uv=edge_curves_uv,

    surface_evaluator=surface_eval_np,
    surface_vertices=surface_vertices,
    surface_faces=surface_faces,

    edge_types=edge_types,
    boundary_edge_mask=boundary_edge_mask,
    boundary_inward_signs=None,
    boundary_strut_is_centered=True,

    # Automatically select boundary-incident blend lengths
    boundary_incident_blend_length=None,

    original_physical_edge_ids=original_physical_edge_ids,
    deduplication_diagnostics=manufacturing_graph_diagnostics,

    tolerance=1e-10,
    spacing_tolerance=1e-3,
)

edge_pass_curves_xyz = np.asarray(
    edge_pass_curves_xyz,
    dtype=np.float64,
)

edge_pass_normals_xyz = np.asarray(
    edge_pass_normals_xyz,
    dtype=np.float64,
)

if edge_pass_curves_uv is not None:
    edge_pass_curves_uv = np.asarray(
        edge_pass_curves_uv,
        dtype=np.float64,
    )

In [11]:
blended_ids = offset_diagnostics[
    "boundary_incident_blended_edge_ids"
]

print("Pass-array shape:", edge_pass_curves_xyz.shape)
print("Boundary-incident blended edges:", blended_ids)

edge0_info = offset_diagnostics[
    "boundary_incident_edge_diagnostics"
].get(0)

print("\nEdge 0 information:")
print(edge0_info)

edge0_spacing = np.linalg.norm(
    edge_pass_curves_xyz[0, 1]
    - edge_pass_curves_xyz[0, 0],
    axis=1,
)

print("\nEdge 0 spacing:")
print("Start     :", edge0_spacing[0])
print("Minimum   :", edge0_spacing.min())
print("Maximum   :", edge0_spacing.max())
print("End       :", edge0_spacing[-1])
print("Requested :", FIBRE_LINE_WIDTH + FIBRE_GAP)

Pass-array shape: (61, 2, 64, 3)
Boundary-incident blended edges: [6, 7, 8, 12, 14, 17, 20, 22, 28, 42]

Edge 0 information:
None

Edge 0 spacing:
Start     : 0.07499572089317949
Minimum   : 0.07499544344835014
Maximum   : 0.07499591696792272
End       : 0.07499590191398867
Requested : 0.07500000000000001


In [12]:
surface_vertices_np = np.asarray(
    surface_vertices,
    dtype=np.float64,
)

surface_faces_np = np.asarray(
    surface_faces,
    dtype=np.int64,
)

# PyVista requires packed faces:
# [3, i0, i1, i2, 3, i0, i1, i2, ...]
if surface_faces_np.ndim == 2:
    packed_faces = np.column_stack(
        (
            np.full(
                len(surface_faces_np),
                3,
                dtype=np.int64,
            ),
            surface_faces_np,
        )
    ).reshape(-1)
else:
    # Already in packed PyVista format
    packed_faces = surface_faces_np.reshape(-1)

mesh = pv.PolyData(
    surface_vertices_np,
    packed_faces,
)

print(mesh)

PolyData (0x72c75fd8d660)
  N Cells:    59058
  N Points:   29858
  N Strips:   0
  X Bounds:   0.000e+00, 1.000e+01
  Y Bounds:   -1.275e-01, 2.101e+00
  Z Bounds:   -2.033e+00, 1.939e+00
  N Arrays:   0


In [13]:
plotter = pv.Plotter()

# Surface
plotter.add_mesh(
    mesh,
    color="lightgray",
    opacity=0.30,
    show_edges=True,
)

# Structural centrelines
for curve in edge_curves_xyz:
    centreline = pv.lines_from_points(
        np.asarray(curve),
        close=False,
    )

    plotter.add_mesh(
        centreline,
        color="gray",
        line_width=1,
        opacity=0.45,
    )

# Manufactured fibre lanes
lane_colours = ["royalblue", "crimson"]

for edge_id in range(
    edge_pass_curves_xyz.shape[0]
):
    for lane_id in range(
        edge_pass_curves_xyz.shape[1]
    ):
        lane_curve = pv.lines_from_points(
            edge_pass_curves_xyz[
                edge_id,
                lane_id,
            ],
            close=False,
        )

        plotter.add_mesh(
            lane_curve,
            color=lane_colours[
                lane_id % len(lane_colours)
            ],
            line_width=3,
        )

# Mark boundary-incident convergence points
blend_points = []

for edge_id, information in offset_diagnostics[
    "boundary_incident_edge_diagnostics"
].items():
    if information["start_boundary_incident"]:
        blend_points.append(
            edge_pass_curves_xyz[
                edge_id, 0, 0
            ]
        )

    if information["end_boundary_incident"]:
        blend_points.append(
            edge_pass_curves_xyz[
                edge_id, 0, -1
            ]
        )

if blend_points:
    blend_points = np.asarray(blend_points)

    plotter.add_mesh(
        pv.PolyData(blend_points),
        color="yellow",
        point_size=10,
        render_points_as_spheres=True,
    )

plotter.add_axes()
plotter.show()

Widget(value='<iframe src="http://localhost:39279/index.html?ui=P_0x72c763d6dd50_2&reconnect=auto" class="pyvi…

In [14]:
# Use the class from the currently reloaded module
EulerCCFToolpath = (
    EulerGraphModule.EulerCCFToolpath
)

# Normalize edge_index for an explicit consistency check
edge_index_euler = np.asarray(
    edge_index,
    dtype=np.int64,
)

if (
    edge_index_euler.ndim == 2
    and edge_index_euler.shape[0] == 2
    and edge_index_euler.shape[1] != 2
):
    edge_index_euler = edge_index_euler.T

physical_edge_count = len(edge_curves_xyz)

assert edge_index_euler.shape == (
    physical_edge_count,
    2,
)

assert edge_pass_curves_xyz.shape[:2] == (
    physical_edge_count,
    fibres_per_edge,
)

assert len(original_physical_edge_ids) == (
    physical_edge_count
)

print("Euler input verification")
print("------------------------")
print("Physical edges     :", physical_edge_count)
print("Fibres per edge    :", fibres_per_edge)
print(
    "Expected traversals:",
    physical_edge_count * fibres_per_edge,
)


tool = EulerCCFToolpath(
    edge_index=edge_index_euler,
    edge_curves_xyz=edge_curves_xyz,
    fibres_per_edge=fibres_per_edge,

    edge_normals_xyz=edge_normals_xyz,
    edge_curves_uv=edge_curves_uv,

    edge_pass_curves_xyz=edge_pass_curves_xyz,
    edge_pass_normals_xyz=edge_pass_normals_xyz,
    edge_pass_curves_uv=edge_pass_curves_uv,
    edge_pass_offsets=edge_pass_offsets,

    original_physical_edge_ids=(
        original_physical_edge_ids
    ),

    physical_edge_metadata={
        "edge_types": np.asarray(edge_types),
        "boundary_edge_mask": np.asarray(
            boundary_edge_mask,
            dtype=bool,
        ),
        "offset_diagnostics": offset_diagnostics,
    },
)

tool.build_multigraph()


tool.build_multigraph()

graph_report = tool.validate_graph(
    verbose=True,
)

baseline_route = tool.compute_euler_circuit()

route_report = tool.validate_route(
    baseline_route,
    geometric_tolerance=1e-5,
    raise_on_error=True,
    verbose=True,
)

print("\nEuler construction complete")
print("---------------------------")
print("Route valid        :", route_report["valid"])
print("Actual traversals  :", len(baseline_route))
print(
    "Expected traversals:",
    physical_edge_count * fibres_per_edge,
)

Euler input verification
------------------------
Physical edges     : 61
Fibres per edge    : 2
Expected traversals: 122
Physical edges       : 61
Multigraph edges     : 122
Graph nodes          : 42
Connected            : True
Odd-degree nodes     : 0
Eulerian             : True
Route traversals       : 122 / 122
Topology errors        : 0
Unique fibre instances : 122 / 122
Multiplicity valid     : True
Maximum endpoint gap   : 0.00000000
Route valid            : True

Euler construction complete
---------------------------
Route valid        : True
Actual traversals  : 122
Expected traversals: 122


In [15]:
manufacturing_settings = (
    EulerCCFToolpath.default_manufacturing_settings(
        endpoint_tolerance=1e-5,

        # Route-ordering limits
        max_junction_turn_degrees=135.0,
        max_internal_bend_degrees=135.0,
        min_revisit_distance=4,

        # Checked later on final manufactured geometry
        min_bend_radius=None,
        min_pass_spacing=None,

        turn_objective=(
            "thresholded_turn_energy"
        ),
        huang_angle_threshold_degrees=60.0,

        infeasible_policy="return_best",
    )
)


route, route_metrics = (
    tool.optimize_turn_aware_euler_circuit(
        trials=500,
        random_seed=42,

        angle_weight=3.0,
        same_edge_penalty=1_000_000.0,
        recent_edge_weight=0.0,
        random_jitter=10.0,
        avoid_same_edge_when_possible=True,

        score_angle_weight=3.0,
        score_turn_120_weight=100.0,
        score_turn_150_weight=500.0,
        score_backtrack_weight=1_000_000.0,
        score_revisit_weight=100.0,
        target_revisit_distance=10,

        turn_objective=(
            "thresholded_turn_energy"
        ),
        huang_angle_threshold_degrees=60.0,

        refinement_passes=5,
        refinement_candidates_per_pass=500,

        manufacturing_settings=(
            manufacturing_settings
        ),
        infeasible_policy="return_best",

        verbose=True,
    )
)

/tmp/ipykernel_3057864/2970431902.py:25: DeprecationWarning: turn_objective='huang_2023' is deprecated; use 'thresholded_turn_energy'. The objective is article-inspired and does not reproduce the article's routing solver.
  tool.optimize_turn_aware_euler_circuit(


Turn-aware trials      : 500
Valid candidates       : 500
Failed candidates      : 0
Best-route updates     : 6
Refinement updates     : 5
Baseline score         : 5771.08
Optimised score        : 1028.03
Baseline feasible      : False (54 violations)
Optimised feasible     : False (4 violations)
No manufacturing-ready route was found for the configured limits. Returning the best Euler-valid route found with labelled violations.

Baseline route
--------------
Mean junction angle    : 86.26
Median junction angle  : 82.70
Maximum junction angle : 180.00
Junctions above 90 deg : 36
Junctions above 120 deg: 30
Junctions above 150 deg: 23
Immediate backtracks   : 23
Maximum internal angle : 0.90
Internal bends >120 deg: 0
Internal bends >150 deg: 0
Maximum full-path angle: 180.00
Full-path turns >120 deg: 30
Full-path turns >150 deg: 23
Mean revisit distance  : 10.31
Minimum revisit distance: 1
Revisit distance = 1   : 23

Turn-aware route
----------------
Mean junction angle    : 63.48
Med

In [16]:
# Explicitly continue from the current best route
tool.euler_edges = [
    dict(item)
    for item in route
]

route, refinement_metrics = (
    tool.optimize_turn_aware_euler_circuit(
        trials=10,
        random_seed=43,

        angle_weight=3.0,
        same_edge_penalty=1_000_000.0,
        recent_edge_weight=0.0,
        random_jitter=10.0,
        avoid_same_edge_when_possible=True,

        score_angle_weight=3.0,
        score_turn_120_weight=100.0,
        score_turn_150_weight=500.0,
        score_backtrack_weight=1_000_000.0,
        score_revisit_weight=100.0,
        target_revisit_distance=10,

        turn_objective=(
            "thresholded_turn_energy"
        ),
        huang_angle_threshold_degrees=60.0,

        # Main change
        refinement_passes=50,
        refinement_candidates_per_pass=500,

        manufacturing_settings=(
            manufacturing_settings
        ),
        infeasible_policy="return_best",

        verbose=True,
    )
)

/tmp/ipykernel_3057864/2400644874.py:8: DeprecationWarning: turn_objective='huang_2023' is deprecated; use 'thresholded_turn_energy'. The objective is article-inspired and does not reproduce the article's routing solver.
  tool.optimize_turn_aware_euler_circuit(


Turn-aware trials      : 10
Valid candidates       : 10
Failed candidates      : 0
Best-route updates     : 0
Refinement updates     : 10
Baseline score         : 1028.03
Optimised score        : 633.69
Baseline feasible      : False (4 violations)
Optimised feasible     : True (0 violations)

Baseline route
--------------
Mean junction angle    : 63.48
Median junction angle  : 48.65
Maximum junction angle : 180.00
Junctions above 90 deg : 14
Junctions above 120 deg: 9
Junctions above 150 deg: 2
Immediate backtracks   : 2
Maximum internal angle : 0.90
Internal bends >120 deg: 0
Internal bends >150 deg: 0
Maximum full-path angle: 180.00
Full-path turns >120 deg: 9
Full-path turns >150 deg: 2
Mean revisit distance  : 31.82
Minimum revisit distance: 1
Revisit distance = 1   : 2

Turn-aware route
----------------
Mean junction angle    : 62.01
Median junction angle  : 48.65
Maximum junction angle : 131.35
Junctions above 90 deg : 14
Junctions above 120 deg: 8
Junctions above 150 deg: 0
Imm

In [17]:
# Preserve the final optimized route explicitly
tool.euler_edges = [
    dict(item)
    for item in route
]

lane_assignment = tool.assign_route_lanes(
    route
)

print("Lane-assignment verification")
print("----------------------------")
print(
    "Assigned traversals:",
    len(lane_assignment),
)
print(
    "Expected traversals:",
    len(route),
)

lane_coverage_errors = []

for edge_id in range(len(edge_index_euler)):
    assigned_lanes = sorted(
        entry["lane_id"]
        for entry in lane_assignment
        if entry["physical_edge_id"] == edge_id
    )

    expected_lanes = list(
        range(fibres_per_edge)
    )

    if assigned_lanes != expected_lanes:
        lane_coverage_errors.append(
            {
                "edge_id": edge_id,
                "assigned": assigned_lanes,
                "expected": expected_lanes,
            }
        )

print(
    "Complete lane coverage:",
    len(lane_coverage_errors) == 0,
)
print(
    "Coverage errors:",
    lane_coverage_errors,
)

print("\nFirst five assignments")
print("----------------------")

for entry in lane_assignment[:5]:
    print(entry)

Lane-assignment verification
----------------------------
Assigned traversals: 122
Expected traversals: 122
Complete lane coverage: True
Coverage errors: []

First five assignments
----------------------
{'traversal_id': 0, 'physical_edge_id': 60, 'edge_id': 60, 'fibre_id': 1, 'fibre_instance_id': 1, 'direction': 'reverse', 'visit_number': 1, 'offset_lane': 0, 'lane_id': 0, 'signed_offset': -0.025}
{'traversal_id': 1, 'physical_edge_id': 17, 'edge_id': 17, 'fibre_id': 1, 'fibre_instance_id': 1, 'direction': 'reverse', 'visit_number': 1, 'offset_lane': 1, 'lane_id': 1, 'signed_offset': 0.037500000000000006}
{'traversal_id': 2, 'physical_edge_id': 16, 'edge_id': 16, 'fibre_id': 1, 'fibre_instance_id': 1, 'direction': 'reverse', 'visit_number': 1, 'offset_lane': 0, 'lane_id': 0, 'signed_offset': -0.037500000000000006}
{'traversal_id': 3, 'physical_edge_id': 14, 'edge_id': 14, 'fibre_id': 0, 'fibre_instance_id': 0, 'direction': 'forward', 'visit_number': 1, 'offset_lane': 0, 'lane_id': 0, 

In [18]:
manufactured_report = tool.validate_manufactured_paths(
    route=route,

    surface_evaluator=surface_eval_np,
    surface_vertices=surface_vertices,
    surface_faces=surface_faces,

    fibre_line_width=FIBRE_LINE_WIDTH,
    strut_thickness=STRUT_THICKNESS,
    boundary_margin=BOUNDARY_MARGIN,

    min_pass_spacing=None,
    min_bend_radius=None,
    max_manufactured_turn_degrees=135.0,

    transition_samples=17,
    junction_trim_distance=FIBRE_LINE_WIDTH,

    require_surface_check=True,
    require_self_intersection_check=True,
    require_lane_crossover_check=True,
    require_unrelated_segment_spacing=False,
    require_junction_bend_radius=False,

    tolerance=1e-5,
    verbose=True,
)

Offset geometry ready : False
Manufacturing ready   : False
Offset violations     : 1


In [ ]:
from collections import Counter, defaultdict

violations = manufactured_report["violations"]

violation_counts = Counter(
    item.get("kind", "unknown")
    for item in violations
)

grouped_violations = defaultdict(list)

for item in violations:
    grouped_violations[
        item.get("kind", "unknown")
    ].append(item)

print("Violation classification")
print("------------------------")

for kind, count in violation_counts.items():
    print(f"{kind:42s}: {count}")

print("\nTransition candidate statistics")
print("-------------------------------")
print(
    manufactured_report[
        "transition_candidate_statistics"
    ]
)

print("\nFailed junctions:",
      manufactured_report[
          "junctions_with_no_valid_transition"
      ])

print("Missing data:",
      manufactured_report["missing_data"])

print("\nExample records")
print("---------------")

for kind, records in grouped_violations.items():
    print(f"\n{kind}")

    for record in records[:3]:
        print(record)

Violation classification
------------------------
manufactured_turn_angle                   : 1

Transition candidate statistics
-------------------------------
{'attempted_transition_candidates': 1952, 'rejected_projection_degenerate_candidates': 44, 'rejected_joined_turn_candidates': 8, 'rejected_radius_candidates': 0, 'junctions_with_no_valid_transition': 0}

Failed junctions: 0
Missing data: []

Example records
---------------

manufactured_turn_angle
{'kind': 'manufactured_turn_angle', 'measured_value': 143.74361440038163, 'limit': 135.0, 'action': 'offset_geometry'}


In [20]:
transition_probe = (
    tool.build_junction_transitions(
        route=route,

        surface_evaluator=surface_eval_np,
        surface_vertices=surface_vertices,
        surface_faces=surface_faces,

        transition_samples=17,
        transition_metric_samples=65,

        transition_tolerance=1e-5,

        # Use a physical trim length instead of one
        # derived from the zero endpoint gap.
        junction_trim_distance=FIBRE_LINE_WIDTH,

        min_bend_radius=None,
        max_joined_turn_degrees=135.0,

        mesh_projection_tolerance=None,
    )
)

print("Transition probe")
print("----------------")
print(
    "Transition count:",
    transition_probe["transition_count"],
)
print(
    "Candidate statistics:",
    transition_probe[
        "candidate_statistics"
    ],
)
print(
    "Violation count:",
    len(transition_probe["violations"]),
)

from collections import Counter

print(
    "Violation types:",
    Counter(
        item["kind"]
        for item in transition_probe["violations"]
    ),
)

Transition probe
----------------
Transition count: 122
Candidate statistics: {'attempted_transition_candidates': 1952, 'rejected_projection_degenerate_candidates': 44, 'rejected_joined_turn_candidates': 8, 'rejected_radius_candidates': 0, 'junctions_with_no_valid_transition': 0}
Violation count: 0
Violation types: Counter()


In [21]:
import importlib
import Utils.EulerGraph as EulerGraphModule

importlib.reload(EulerGraphModule)

EulerCCFToolpath = (
    EulerGraphModule.EulerCCFToolpath
)

# Reconstruct using the updated class
tool = EulerCCFToolpath(
    edge_index=edge_index_euler,
    edge_curves_xyz=edge_curves_xyz,
    fibres_per_edge=fibres_per_edge,

    edge_normals_xyz=edge_normals_xyz,
    edge_curves_uv=edge_curves_uv,

    edge_pass_curves_xyz=edge_pass_curves_xyz,
    edge_pass_normals_xyz=edge_pass_normals_xyz,
    edge_pass_curves_uv=edge_pass_curves_uv,
    edge_pass_offsets=edge_pass_offsets,

    original_physical_edge_ids=(
        original_physical_edge_ids
    ),

    physical_edge_metadata={
        "edge_types": np.asarray(edge_types),
        "boundary_edge_mask": np.asarray(
            boundary_edge_mask,
            dtype=bool,
        ),
        "offset_diagnostics": offset_diagnostics,
    },
)

tool.build_multigraph()

# Reuse the optimized route
tool.euler_edges = [
    dict(item)
    for item in route
]

tool.validate_route(
    route,
    geometric_tolerance=1e-5,
    raise_on_error=True,
    verbose=True,
)

Route traversals       : 122 / 122
Topology errors        : 0
Unique fibre instances : 122 / 122
Multiplicity valid     : True
Maximum endpoint gap   : 0.00000000
Route valid            : True


{'valid': True,
 'traversal_count_valid': True,
 'topology_valid': True,
 'topology_errors': [],
 'unique_instances_valid': True,
 'multiplicity_valid': np.True_,
 'visit_counts': array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]),
 'geometry_valid': np.True_,
 'geometric_gaps': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.])}

In [22]:
manufactured_report = (
    tool.validate_manufactured_paths(
        route=route,

        surface_evaluator=surface_eval_np,
        surface_vertices=surface_vertices,
        surface_faces=surface_faces,

        fibre_line_width=FIBRE_LINE_WIDTH,
        strut_thickness=STRUT_THICKNESS,
        boundary_margin=BOUNDARY_MARGIN,

        min_pass_spacing=None,
        min_bend_radius=None,
        max_manufactured_turn_degrees=135.0,

        transition_samples=17,

        # The newly propagated parameter
        junction_trim_distance=FIBRE_LINE_WIDTH,

        require_surface_check=True,
        require_self_intersection_check=True,
        require_lane_crossover_check=True,
        require_unrelated_segment_spacing=False,
        require_junction_bend_radius=False,

        tolerance=1e-5,
        verbose=True,
    )
)

Offset geometry ready : False
Manufacturing ready   : False
Offset violations     : 1


In [23]:
from collections import Counter, defaultdict

violations = manufactured_report["violations"]

counts = Counter(
    item.get("kind", "unknown")
    for item in violations
)

grouped = defaultdict(list)

for item in violations:
    grouped[
        item.get("kind", "unknown")
    ].append(item)

print("Remaining violation classification")
print("----------------------------------")

for kind, count in counts.items():
    print(f"{kind:42s}: {count}")

print("\nTransition status")
print("-----------------")
print(
    "Failed junctions:",
    manufactured_report[
        "junctions_with_no_valid_transition"
    ],
)
print(
    "Candidate statistics:",
    manufactured_report[
        "transition_candidate_statistics"
    ],
)

print("\nViolation records")
print("-----------------")

for kind, records in grouped.items():
    print(f"\n{kind}")

    for record in records:
        print(record)

Remaining violation classification
----------------------------------
manufactured_turn_angle                   : 1

Transition status
-----------------
Failed junctions: 0
Candidate statistics: {'attempted_transition_candidates': 1952, 'rejected_projection_degenerate_candidates': 44, 'rejected_joined_turn_candidates': 8, 'rejected_radius_candidates': 0, 'junctions_with_no_valid_transition': 0}

Violation records
-----------------

manufactured_turn_angle
{'kind': 'manufactured_turn_angle', 'measured_value': 143.74361440038163, 'limit': 135.0, 'action': 'offset_geometry'}


In [ ]:
from pathlib import Path
from IPython.display import Video, display

VIDEO_DIR = BASE / "ToolPath_Output"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

video_path = VIDEO_DIR / "manufactured_offset_toolpath.avi"

# Ensure the optimized route is active
tool.euler_edges = route

saved_video = tool.Generate_Video_Offsets(
    filename=str(video_path),
    fps=15,
    width=1400,
    height=900,
    samples_per_edge=20,
    hold_frames=1,
)

print("Saved video:", saved_video)

display(
    Video(
        str(video_path),
        embed=True,
        width=900,
    )
)

Saved: /home/arash/HVD_SeedsBase/ToolPath_Output/manufactured_offset_toolpath.avi
Saved video: /home/arash/HVD_SeedsBase/ToolPath_Output/manufactured_offset_toolpath.avi
